# Urjo growth analysis — 2026-07-15

Reproducible audit of the two Reddit/Devvit exports supplied for r/urjo. Counts are treated as daily aggregates or events—not unique users—unless the source explicitly says otherwise. Release attribution is unknown and no paid advertising ran during the period.

In [ ]:
from pathlib import Path
import pandas as pd

ANALYTICS_PATH = Path('/Users/vigneshaithal/Downloads/Urjo_analytics_20260715_050725.csv')
JOURNEYS_PATH = Path('/Users/vigneshaithal/Downloads/Urjo_journeys_analytics_20260715_050730.csv')

# The analytics header is malformed: commas inside four labels are not quoted.
analytics_columns = [
    'date', 'qualified_installs', 'qualified_engagers', 'logged_in', 'logged_out',
    'qe_7d_avg', 'qe_7d_logged_in', 'qe_7d_logged_out',
    'qe_14d_avg', 'qe_14d_logged_in', 'qe_14d_logged_out', 'tier',
]
analytics = pd.read_csv(ANALYTICS_PATH, skiprows=1, names=analytics_columns)
analytics['date'] = pd.to_datetime(analytics['date'])
analytics = analytics.sort_values('date').reset_index(drop=True)
journeys = pd.read_csv(JOURNEYS_PATH)
journeys['utc_day'] = pd.to_datetime(journeys['utc_day'])
analytics.head(3), journeys.head(3)

In [ ]:
quality = {
    'analytics_rows': len(analytics),
    'analytics_date_min': analytics['date'].min().date().isoformat(),
    'analytics_date_max': analytics['date'].max().date().isoformat(),
    'analytics_missing_dates': len(pd.date_range(analytics['date'].min(), analytics['date'].max()).difference(analytics['date'])),
    'analytics_duplicate_dates': int(analytics['date'].duplicated().sum()),
    'journey_rows': len(journeys),
    'journey_date_min': journeys['utc_day'].min().date().isoformat(),
    'journey_date_max': journeys['utc_day'].max().date().isoformat(),
    'journey_duplicate_dates': int(journeys['utc_day'].duplicated().sum()),
}
pd.Series(quality, name='value').to_frame()

In [ ]:
first_14 = analytics[(analytics['date'] >= '2026-06-16') & (analytics['date'] <= '2026-06-29')]
second_14 = analytics[(analytics['date'] >= '2026-06-30') & (analytics['date'] <= '2026-07-13')]
previous_week = analytics[(analytics['date'] >= '2026-06-30') & (analytics['date'] <= '2026-07-06')]
latest_week = analytics[(analytics['date'] >= '2026-07-07') & (analytics['date'] <= '2026-07-13')]
comparable_journeys = journeys[journeys['utc_day'] >= '2026-07-10']

headline = {
    '30d_qe_user_days': int(analytics['qualified_engagers'].sum()),
    '30d_daily_qe_mean': analytics['qualified_engagers'].mean(),
    '30d_daily_qe_median': analytics['qualified_engagers'].median(),
    '14d_prior_daily_avg': first_14['qualified_engagers'].mean(),
    '14d_latest_daily_avg': second_14['qualified_engagers'].mean(),
    '14d_change': second_14['qualified_engagers'].mean() / first_14['qualified_engagers'].mean() - 1,
    'latest_7d_daily_avg': latest_week['qualified_engagers'].mean(),
    'latest_7d_change': latest_week['qualified_engagers'].mean() / previous_week['qualified_engagers'].mean() - 1,
    'latest_7d_cv': latest_week['qualified_engagers'].std(ddof=1) / latest_week['qualified_engagers'].mean(),
    'logged_out_share_30d': analytics['logged_out'].sum() / analytics['qualified_engagers'].sum(),
    'journey_event_start_rate_jul10_13': comparable_journeys['journey_start_count'].sum() / comparable_journeys['app_ready_count'].sum(),
    'journey_end_per_start_jul10_13': comparable_journeys['journey_end_count'].sum() / comparable_journeys['journey_start_count'].sum(),
}
pd.Series(headline, name='value').to_frame()

In [ ]:
weekly = (analytics.assign(week_start=analytics['date'] - pd.to_timedelta(analytics['date'].dt.dayofweek, unit='D'))
          .groupby('week_start', as_index=False)
          .agg(days=('date', 'size'), average_daily_qe=('qualified_engagers', 'mean'), total_qe_days=('qualified_engagers', 'sum')))
weekly

## Interpretation guardrails

- Summed Qualified Engagers are daily user-days, not unique people.
- Journey counts are repeated events and cannot be treated as a person-level funnel.
- July 9 is likely partial or an instrumentation boundary and is excluded from the comparable journey aggregate.
- The export cannot identify acquisition source, D1/D7 retention, per-post CTR, K-factor, or causal impact of a release.